# Overview

**Single-Agent Fusion (Blind) Pipeline**

This notebook implements the a single-agent fusion pipeline for the Fake News Detection project.  
In this setup, the LLM agent **cannot see the image directly**. Instead, it relies on:

- the human-provided **headline**,  
- an AI-generated **image description**, and  
- a set of **forensic tools**

to decide whether a news item is real or fake.

The goal of this notebook is to:

- Evaluate how well a structured, tool-using LLM agent can detect fake news without direct visual access  
- Compare the agent's decisions to unimodal baselines:
  - CLIP zero-shot image classifier (vision-only)
  - RoBERTa fake-news classifier on headlines and captions (text-only)
- Study the effect of fusion by comparing:
  - the agent's **initial impression** (headline-based only)
  - the agent's **final decision** after consulting tools  
- Analyze when tool fusion helps, when it hurts, and how often the agent changes its mind

This pipeline exposes two levels of prediction:

1. **Tool-Level Predictions (Baselines)**  
   - **Vision Tool (CLIP):**  
     Zero-shot `openai/clip-vit-base-patch32` predicts Fake/Real from the image and extracts top-k visual concepts.  
   - **Text Tool (RoBERTa + VADER):**  
     A Fakeddit-trained RoBERTa classifier (`yaoyinnan/roberta-fakeddit`) labels the headline as Fake/Real with a confidence score, and VADER computes a sentiment score (e.g., for sensationalism).  
   - **Consistency Tool (MPNet + CLIP):**  
     `all-mpnet-base-v2` measures semantic similarity between headline and BLIP-2 caption (generated by `Salesforce/blip2-opt-2.7b`), and CLIP computes cross-modal similarity between the image and both texts.

2. **Agent-Level Decisions (LLM Reasoning)**  
   - `gpt-4.1-mini` is used as a LangChain legacy agent (`AgentExecutor` + `create_tool_calling_agent`).  
   - It follows a **phase-based protocol**:
     1. Form an initial impression (Real/Fake) from the headline only  
     2. Check the reliability of the AI image description  
     3. Call the vision, text, and consistency tools as "forensic sensors"  
     4. Compare intuition vs tool evidence and output a final decision  
   - The agent returns:
     - `Initial_Impression` and `Final_Decision` (Fake / Real)  
     - `Did_Decision_Change` (Yes / No) and `Decision_Confidence` (High/Medium/Low)  
     - A detailed `Explanation` and a user-facing `Nudge`  

---

**What This Notebook Covers**

- Load:  
\- The preprocessed Fakeddit subset and images (N=225)  
\- BLIP-2 captions (precomputed in the metadata)  
\- Visual concepts embeddings (precomputed)  
- Define the three LangChain tools (vision, text, consistency)
- Compute Top-K visual concepts using ImageNet + Places365 concept embeddings
- Perform CLIP zero-shot real/fake classification
- Classify both headline and BLIP captions using RoBERTa
- Perform sentiment analysis using VADER
- Compute semantic similarity between headline and caption using MPNet SentenceTransformer
- Compute cross-modal similarity between image and texts (headline+caption) using CLIP
- Build the blind agent (GPT) with a structured system prompt and a phase-based workflow
- Fuse all signals using GPT-4.1-mini for reasoning
- Evaluate performances (CLIP, RoBERTa, GPT)

> This notebook serves as an intermediate LangChain-based fusion baseline before building the full multimodal agentic system.

# Setup (LangChain Legacy Agents Compatibility Fix)

This notebook uses **LangChain legacy agents** (e.g. `AgentExecutor`, `create_tool_calling_agent`), which are **not compatible with the latest LangChain releases** shipped in Colab by default.

To avoid version conflicts:

1. Uninstall any preinstalled `numpy`, `pandas`, and `langchain*` packages.
2. Install a set of versions that are known to work with the legacy agents used in this notebook.

> **Important:** After running the two cells below,   
> you **must restart the runtime** (`Runtime → Restart session`) before continuing.  
> If you do not restart, Python may still use the old in-memory versions, and imports / agents can fail in confusing ways.

In [ ]:
# Uninstall the conflicting packages to start fresh
!pip uninstall -y numpy pandas langchain langchain-core langchain-community langchain-openai


In [ ]:
# Install the exact compatible stack for Colab + Legacy Agents
!pip install -q "numpy==1.26.4" \
                "pandas==2.2.2" \
                "scikit-learn" \
                "langchain==0.3.0" \
                "langchain-core==0.3.0" \
                "langchain-community==0.3.0" \
                "langchain-openai==0.2.0" \
                "vaderSentiment"

**IMPORTANT REMINDER:** Now restart the runtime (Runtime → Restart session) before running the next cells.

# Setup (Dependencies & Imports)

Install dependencies, clone the repo, and import required libraries and modules.

In [ ]:
# Clone the project repo
!git clone https://github.com/gizayceylan/FakeNews.git

# Add it to Python path
import sys
sys.path.append("/content/FakeNews")


In [ ]:
# Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
import json
import time
import subprocess

from tqdm import tqdm
from PIL import Image
from numpy.linalg import norm

# ML/DL libraries
from transformers import CLIPProcessor, CLIPModel
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer, util
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.metrics import confusion_matrix, classification_report

# LangChain
from langchain.tools import tool
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Import custom utility for OpenAI Client
from utils.api_key import load_openai_client

In [ ]:
# Set OpenAI API
client = load_openai_client()

# Metadata (Dataset Loading & Preparation)

Load the Fakeddit metadata, clean it, and select the subset of samples containing valid image URLs.

In [ ]:
# Unzip fakeddit_images.zip from /content/FakeNews/assets/content/master_pipeline_assets/
!unzip -q /content/FakeNews/assets/fakeddit_images.zip -d /content/FakeNews/assets

In [ ]:
# Load the pre-processed metadata
subset_path = "/content/FakeNews/assets/fakeddit_balanced_subset.csv"
image_df = pd.read_csv(subset_path)

# Define the path for the image directory (which was unzipped)
img_dir = "/content/FakeNews/assets/content/fakeddit_images"

# Update the DataFrame paths to reflect the new location
image_df['image_path'] = image_df['image_path'].apply(
    lambda p: os.path.join(img_dir, os.path.basename(p))
)

print(f"Loaded {len(image_df)} clean samples.")
print("Shape subset:", image_df.shape)
print(image_df["label"].value_counts())
image_df.head()


In [ ]:
# Test a sample
Image.open(image_df["image_path"].iloc[0])


In [ ]:
# Load visual concepts list
assets_dir = "/content/FakeNews/assets"
with open(os.path.join(assets_dir, "concepts_list.json"), "r") as f:
    concepts_list = json.load(f)

# Load visual concept embeddings
concept_embs = np.load(os.path.join(assets_dir, "concept_embeddings.npy"))
print(f"Loaded {len(concepts_list)} concepts and embedding matrix {concept_embs.shape}")

# Model & Tools

Lood all the models and define the LangChain tools.

In [ ]:
# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Log GPU specs (model, driver, VRAM)
try:
    smi = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
        text=True
    ).strip()
    gpu_name, driver_version, vram_total = [x.strip() for x in smi.split(",")]
except Exception:
    gpu_name, driver_version, vram_total = "CPU/Unknown", None, None

In [ ]:
# CLIP
clip_name = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(clip_name)
clip_processor = CLIPProcessor.from_pretrained(clip_name)
clip_model.to(device)
clip_model.eval()
print("\n-----------------------------------------------------")
print("CLIP loaded successfully.")
print("-----------------------------------------------------")

# RoBERTa
roberta_name = "yaoyinnan/roberta-fakeddit"
roberta_tok = AutoTokenizer.from_pretrained(roberta_name)
roberta_model = AutoModelForSequenceClassification.from_pretrained(roberta_name)
roberta_model.to(device)
roberta_model.eval()
print("\n-----------------------------------------------------")
print("RoBERTa loaded successfully.")
print("-----------------------------------------------------")

# SentenceTransformer
sim_model = SentenceTransformer("all-mpnet-base-v2")
sim_model.to(device)
print("\n-----------------------------------------------------")
print("MPNet loaded successfully.")
print("-----------------------------------------------------")

# Sentiment Heuristic (VADER)
vader_analyzer = SentimentIntensityAnalyzer()

print("\n-----------------------------------------------------")
print("All models loaded successfully.")
print("-----------------------------------------------------")


## Vision Tool

Gather all visual evidence through CLIP.

In [ ]:
# Define labels for CLIP prompts
label_prompts = [
    "a staged or manipulated fake news photo",
    "a real photo from a legitimate news report"
]

In [ ]:
@tool
def vision_tool(image_path: str) -> dict:
    """
    Analyzes an image file.
    Returns: top visual concepts (CLIP), and Fake/Real classification (CLIP Zero-Shot).
    """
    image = Image.open(image_path).convert("RGB")

    # CLIP features (for concepts and zero-shot)
    inputs = clip_processor(images=image, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        img_emb = clip_model.get_image_features(**inputs)
    img_vec = img_emb.squeeze().cpu().numpy()

    # 1. Top-K Concepts
    k=5
    img_vec_norm = img_vec / norm(img_vec)
    concept_mags = norm(concept_embs, axis=1)
    concept_embs_norm = concept_embs / concept_mags[:, np.newaxis]
    sims = concept_embs_norm @ img_vec_norm
    topk_idx = sims.argsort()[::-1][:k]
    topk = [f"{concepts_list[i]} ({sims[i]:.2f})" for i in topk_idx]

    # 2. CLIP Zero-Shot
    text_inputs = clip_processor(text=label_prompts, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        text_embs = clip_model.get_text_features(**text_inputs)

    # Calculate similarity for ZS (manual dot product)
    text_embs_norm = text_embs / text_embs.norm(dim=1, keepdim=True)
    img_emb_norm = img_emb / img_emb.norm(dim=1, keepdim=True)
    logits = (img_emb_norm @ text_embs_norm.T).squeeze()
    probs = logits.softmax(dim=0).cpu().numpy()

    zs_pred_idx = np.argmax(probs)
    zs_label = "Real" if zs_pred_idx == 1 else "Fake"
    zs_conf = float(probs[zs_pred_idx])

    return {

            "visual_concepts": topk,
            "clip_zs_label": zs_label,
            "clip_zs_confidence": zs_conf
    }


## Text Tool

Gather all textual evidence through RoBERTa and VADER.

In [ ]:
@tool
def text_tool(user_text: str, AI_caption: str) -> dict:
    """
    Analyzes BOTH the User Headline and the AI Caption in one batch.
    Returns: RoBERTa classification and VADER scores for both texts.
    """

    results = {}

    # Process both texts in a loop to avoid code duplication
    inputs = {"user_text": user_text, "AI_caption": AI_caption}

    for key, text_content in inputs.items():
        # 1. RoBERTa Classification
        tokens = roberta_tok(text_content, return_tensors="pt", truncation=True).to(device)
        with torch.no_grad():
            out = roberta_model(**tokens)
            probs = F.softmax(out.logits, dim=1).squeeze()

        pred_idx = torch.argmax(probs).item()
        label = "Fake" if pred_idx == 0 else "Real"
        conf = float(probs[pred_idx])

        # 2. VADER Sentiment
        sentiment = vader_analyzer.polarity_scores(text_content)["compound"]

        # Store results with specific keys
        results[f"{key}_roberta_label"] = label
        results[f"{key}_roberta_conf"] = conf
        results[f"{key}_vader_sentiment"] = sentiment

    return results


## Consistency Tool

Gather all multimodal alignment evidence through MPNet and CLIP.

In [ ]:
@tool
def consistency_tool(user_text: str, AI_caption: str, image_path: str) -> dict:
    """
    Computes three-way alignment scores:
    1. Semantic: Text vs Caption
    2. Cross-modal: Image vs Text (User Claim Alignment)
    3. Cross-modal: Image vs Caption (BLIP-2 Fidelity Check)
    """
    # 1. Semantic Similarity (MPNet - Text-to-Text)
    emb1 = sim_model.encode(user_text, convert_to_tensor=True, device=device)
    emb2 = sim_model.encode(AI_caption, convert_to_tensor=True, device=device)
    st_sim = util.cos_sim(emb1, emb2).item()  # in [-1, 1]

    # 2. Cross-Modal Similarity (CLIP - Image-to-Text & Image-to-Caption)
    image = Image.open(image_path).convert("RGB")

    # Get separate image & text features
    image_inputs = clip_processor(images=image, return_tensors="pt").to(device)
    text_inputs = clip_processor(text=[user_text, AI_caption], return_tensors="pt", padding=True).to(device)

    with torch.no_grad():
        img_feats = clip_model.get_image_features(**image_inputs)
        txt_feats = clip_model.get_text_features(**text_inputs)

    # Normalize to unit length
    img_feats = img_feats / img_feats.norm(dim=-1, keepdim=True)
    txt_feats = txt_feats / txt_feats.norm(dim=-1, keepdim=True)

    # Cosine similarity: dot product of normalized vectors
    sims = (img_feats @ txt_feats.T).squeeze()  # shape (2,), values in [-1, 1]

    # image vs headline = index 0, image vs caption = index 1
    img_text_sim = float(sims[0].item())
    img_cap_sim  = float(sims[1].item())

    return {
        "text_to_caption_similarity": st_sim,
        "image_to_text_similarity": img_text_sim,
        "image_to_caption_similarity": img_cap_sim
    }


# Agentic Reasoning (GPT Fusion Layer)

Provide reasoning and final decision through GPT.

In [ ]:
# Agent setup
tools = [vision_tool, text_tool, consistency_tool]

# Define the System Prompt
system_prompt = """
You are a Misinformation Detection Expert Analyst.
Your goal is to distinguish between **Malicious Fake News** and **Benign Content**.
You do not have direct access to the image of news but you can use your tools to analyze it.
Therefore, you must rely on:
- the HEADLINE,
- the AI_IMAGE_DESCRIPTION,
- and the outputs of your forensic tools.

CORE PHILOSOPHY:
Form an initial judgment from the headline alone, then use tools as your "forensic senses" for the hidden image and deeper signals.

YOUR TOOLKIT:
1. Vision Tool:
   - Detects visual concepts in the image.
   - Flags the image as potentially Fake or Real using a vision classifier.
2. Text Tool:
   - Analyzes BOTH the user-provided headline and the AI-generated image description.
   - Flags each as potentially Fake or Real using a fake-news classifier.
   - Measures sentiment (e.g., sensationalism) using sentiment analysis.
3. Consistency Tool:
   - Measures how well the image, AI description, and headline align using semantic and cross-modal similarity:
     - text_to_caption_similarity: cosine similarity between headline and AI caption (range [-1, 1]).
     - image_to_text_similarity: cosine similarity between image and headline (range [-1, 1]).
     - image_to_caption_similarity: cosine similarity between image and AI caption (range [-1, 1]).

IMPORTANT TOOL RULES:
- Before giving your final answer, you MUST call:
  - vision_tool
  - text_tool
  - consistency_tool
- Use the actual tool outputs as evidence. Do NOT invent or fabricate tool results.
- You may change your initial intuition if the tools provide strong contradictory evidence.

DATA GATHERING PROTOCOL:

PHASE 1: DIRECT OBSERVATION (System 1 Intuition)
- For your initial impression, you MUST rely ONLY on the provided HEADLINE.
- IGNORE the AI image description in this phase.
- Then choose your initial impression: "Real" or "Fake", and explain briefly WHY based on what you READ in the headline.

PHASE 2: IMAGE DESCRIPTION RELIABILITY (Reliability Check)
- You are provided with an AI-generated Image Description.
- Compare this description with the HEADLINE.
- Is the description aligned with the headline?
  - If YES: You may treat the description as a potentially useful complement to the headline.
  - If NO: The description might be hallucinated or vague, treat it with caution in later reasoning.

PHASE 3: FORENSIC CONSULTATION (System 2 Forensics)
- Call vision_tool to:
  - Check which visual concepts are detected.
  - See whether the image is predicted as Fake or Real, and with what confidence.
- Call text_tool to:
  - Check whether the headline and AI caption are predicted as Fake or Real, and with what confidence.
  - Inspect their sentiment scores (e.g., very extreme or sensational tone).
- Call consistency_tool to:
  - Measure alignment between:
    - Headline and AI caption (semantic similarity).
    - Image and Headline (cross-modal similarity).
    - Image and AI caption (cross-modal similarity).
  - High values near 1.0 indicate strong alignment, values near 0 indicate weak or generic alignment, and values below 0 indicate contradiction.

PHASE 4: SYNTHESIS (The Verdict)
- Compare Phase 1 (Intuition) with Phase 3 (Forensics).
- Check whether the tools found something you missed.
- Reflect explicitly: did the tools change your mind?
- Then choose a final classification: "Real" or "Fake".

CONFIDENCE FIELD:
When setting "Decision_Confidence":
- Use "High" if most tools strongly agree with each other and with your final decision.
- Use "Medium" if the evidence is mixed but clearly leans toward your final decision.
- Use "Low" if the evidence is weak or conflicting and you had to choose despite uncertainty.

OUTPUT FORMAT:
Return a single JSON object (no markdown, no extra commentary), with this exact structure:

{{
  "Initial_Impression": "Real" or "Fake",
  "Initial_Reasoning": "Brief summary of your initial impression.",
  "Final_Decision": "Real" or "Fake",
  "Did_Decision_Change": "Yes" or "No",
  "Decision_Confidence": "High" or "Medium" or "Low",
  "Explanation": "Detailed narrative. Start with your intuition, mention whether the AI description was reliable, and explain how the tool evidence confirmed or refuted your initial thought.",
  "Nudge": "One actionable piece of advice for the user about how to think more critically about similar content in the future."
}}

"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.0) # or gpt-4o-mini
agent = create_tool_calling_agent(llm, tools, prompt)


In [ ]:
# Make Verbose=False to keep output clean
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=False)

# Run pipeline
results = []

# Start timer
T0 = time.perf_counter()

# Iterate through the DataFrame
for i, row in tqdm(image_df.iterrows(), total=len(image_df)): # use sample_df for N samples
    current_image_path = row['image_path']
    current_headline = row['clean_title']
    current_AIcaption = row['blip2_caption']
    true_label = row['label']

    # Create Structured Input (Text)
    user_input = [
        {
            "type": "text",
            "text": f"""
Please analyze this news item using your tools.

HEADLINE: "{current_headline}"
AI_IMAGE_DESCRIPTION: "{current_AIcaption}"
LOCAL_IMAGE_PATH: "{current_image_path}"

Use this LOCAL_IMAGE_PATH string exactly when calling:
- vision_tool(image_path=LOCAL_IMAGE_PATH)
- consistency_tool(user_text=HEADLINE, AI_caption=AI_IMAGE_DESCRIPTION, image_path=LOCAL_IMAGE_PATH)
            """.strip()
        }
    ]

    try:
        # Run the Agent
        response = agent_executor.invoke({"input": user_input})

        # Extract output
        output_text = response['output']

        # Clean markdown code blocks if present
        clean_json = output_text.replace("```json", "").replace("```", "").strip()

        # Parse JSON output
        decision_data = json.loads(clean_json)

        # Baseline predictions (CLIP & RoBERTa)
        vision_res = vision_tool.invoke({"image_path": current_image_path})
        text_res   = text_tool.invoke({"user_text": current_headline, "AI_caption": current_AIcaption})

        # Store all fields
        results.append({
            "image_path": current_image_path,
            "headline": current_headline,
            "true_label": true_label,
            "blip2_caption": current_AIcaption,

            # Tool predictions
            "clip_label": vision_res.get("clip_zs_label"),
            "roberta_text_label": text_res.get("user_text_roberta_label"),
            "roberta_cap_label": text_res.get("AI_caption_roberta_label"),

            # Decision fields
            "initial_impression": decision_data.get("Initial_Impression"),
            "initial_reasoning": decision_data.get("Initial_Reasoning"),
            "final_decision": decision_data.get("Final_Decision"),
            "changed_mind": decision_data.get("Did_Decision_Change"),
            "confidence": decision_data.get("Decision_Confidence"),

            # Explanation
            "explanation": decision_data.get("Explanation"),
            "nudge": decision_data.get("Nudge"),

            # Debugging
            "raw_output": output_text
        })

    except Exception as e:
        print(f"Error on row {i}: {e}")
        results.append({
            "image_path": current_image_path,
            "headline": current_headline,
            "error": str(e)
        })

# Stop timer
T1 = time.perf_counter()
print(f"\nTotal runtime: {(T1 - T0)/60:.2f} min  ({T1 - T0:.1f} sec)")

# Save results
results_df = pd.DataFrame(results)
print("\nProcessing Complete.")
results_df.head()


# Detailed Evaluation (Performance Comparisons)

Compare classification performances: CLIP, RoBERTa, GPT pre-fuse, GPT post-fuse.

In [ ]:
# Convert textual labels to ints
def label_to_int(x):
    if str(x).lower() == "fake":  return 0
    if str(x).lower() == "real":  return 1
    return None  # for errors / missing / unexpected values

# Make a new df for evaluation
eval_df = pd.DataFrame({
    "true_label": results_df["true_label"].astype(int),
    "clip_label": results_df["clip_label"].apply(label_to_int),
    "robertaT_label": results_df["roberta_text_label"].apply(label_to_int),
    "robertaC_label": results_df["roberta_cap_label"].apply(label_to_int),
    "gpt_pre":  results_df["initial_impression"].apply(label_to_int),
    "gpt_post": results_df["final_decision"].apply(label_to_int),
})


In [ ]:
# Accuracies
clip_acc     = np.mean(eval_df["clip_label"]     == eval_df["true_label"])
robertaT_acc = np.mean(eval_df["robertaT_label"] == eval_df["true_label"])
robertaC_acc = np.mean(eval_df["robertaC_label"] == eval_df["true_label"])
gpt1_acc     = np.mean(eval_df["gpt_pre"]        == eval_df["true_label"])
gpt2_acc     = np.mean(eval_df["gpt_post"]       == eval_df["true_label"])

print("CLIP (Vision) accuracy:", round(clip_acc, 3))
print("RoBERTa (Text-Headline) accuracy:", round(robertaT_acc, 3))
print("RoBERTa (Text-Caption) accuracy:", round(robertaC_acc, 3))
print("GPT (Pre-Fuse) accuracy:", round(gpt1_acc, 3))
print("GPT (Post-Fuse) accuracy:", round(gpt2_acc, 3))


In [ ]:
# Confusion matrices

# CLIP CM
clip_cm = confusion_matrix(eval_df["true_label"], eval_df["clip_label"])

# RoBERTa CM
robertaT_cm = confusion_matrix(eval_df["true_label"], eval_df["robertaT_label"])
robertaC_cm = confusion_matrix(eval_df["true_label"], eval_df["robertaC_label"])

# GPT pre-fuse CM
gpt1_cm = confusion_matrix(eval_df["true_label"], eval_df["gpt_pre"])

# GPT post-fuse CM
gpt2_cm = confusion_matrix(eval_df["true_label"], eval_df["gpt_post"])

print("\nCLIP (Vision) Confusion Matrix:\n", clip_cm)
print("\nRoBERTa (Text-Headline) Confusion Matrix:\n", robertaT_cm)
print("\nRoBERTa (Text-Caption) Confusion Matrix:\n", robertaC_cm)
print("\nGPT (Pre-Fuse) Confusion Matrix:\n", gpt1_cm)
print("\nGPT (Post-Fuse) Confusion Matrix:\n", gpt2_cm)


In [ ]:
# Heatmaps
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
axes = axes.ravel()  # flatten to 1D: axes[0]..axes[5]

sns.heatmap(clip_cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Pred Fake", "Pred Real"],
            yticklabels=["True Fake", "True Real"],
            ax=axes[0])
axes[0].set_title("CLIP (Vision) Confusion Matrix")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")

sns.heatmap(robertaT_cm, annot=True, fmt="d", cmap="Purples",
            xticklabels=["Pred Fake", "Pred Real"],
            yticklabels=["True Fake", "True Real"],
            ax=axes[1])
axes[1].set_title("RoBERTa (Text-Headline) Confusion Matrix")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("True")

sns.heatmap(robertaC_cm, annot=True, fmt="d", cmap="Purples",
            xticklabels=["Pred Fake", "Pred Real"],
            yticklabels=["True Fake", "True Real"],
            ax=axes[2])
axes[2].set_title("RoBERTa (Text-Caption) Confusion Matrix")
axes[2].set_xlabel("Predicted")
axes[2].set_ylabel("True")

sns.heatmap(gpt1_cm, annot=True, fmt="d", cmap="Greens",
            xticklabels=["Pred Fake", "Pred Real"],
            yticklabels=["True Fake", "True Real"],
            ax=axes[3])
axes[3].set_title("GPT (Pre-Fuse) Confusion Matrix")
axes[3].set_xlabel("Predicted")
axes[3].set_ylabel("True")

sns.heatmap(gpt2_cm, annot=True, fmt="d", cmap="Greens",
            xticklabels=["Pred Fake", "Pred Real"],
            yticklabels=["True Fake", "True Real"],
            ax=axes[4])
axes[4].set_title("GPT (Post-Fuse) Confusion Matrix")
axes[4].set_xlabel("Predicted")
axes[4].set_ylabel("True")

axes[5].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Classification reports
print("\n-----------------------------------------------------")
print("CLIP (Vision) Classification Report:")
print("-----------------------------------------------------")
print(classification_report(
    eval_df["true_label"],
    eval_df["clip_label"],
    target_names=["Fake", "Real"]
))

print("\n-----------------------------------------------------")
print("RoBERTa (Text-Headline) Classification Report:")
print("-----------------------------------------------------")
print(classification_report(
    eval_df["true_label"],
    eval_df["robertaT_label"],
    target_names=["Fake", "Real"]
))

print("\n-----------------------------------------------------")
print("RoBERTa (Text-Caption) Classification Report:")
print("-----------------------------------------------------")
print(classification_report(
    eval_df["true_label"],
    eval_df["robertaC_label"],
    target_names=["Fake", "Real"]
))

print("\n-----------------------------------------------------")
print("GPT (Pre-Fuse) Classification Report:")
print("-----------------------------------------------------")
print(classification_report(
    eval_df["true_label"],
    eval_df["gpt_pre"],
    target_names=["Fake", "Real"]
))

print("\n-----------------------------------------------------")
print("GPT (Post-Fuse) Classification Report:")
print("-----------------------------------------------------")
print(classification_report(
    eval_df["true_label"],
    eval_df["gpt_post"],
    target_names=["Fake", "Real"]
))

# Log

In [ ]:
# Show runtime
runtime_sec = T1 - T0
print(f"Total runtime: {runtime_sec/60:.2f} min ({runtime_sec:.1f} sec)")

# Show GPU specs
display(gpu_name, driver_version, vram_total)

In [ ]:
# Log runtime and gpu specs
log = {
    "pipeline": "Single_Agent_Fusion_Blind",
    "n_samples": len(image_df),
    "runtime_sec": runtime_sec,
    "gpu_name": gpu_name,
    "driver_version": driver_version,
    "vram_total": vram_total,
}
print(log)

# Save

In [ ]:
# Save logs
with open("SAFb_runtime_log.jsonl", "a") as f:
    f.write(json.dumps(log) + "\n")

In [ ]:
# Save results_df
results_df.to_csv("SAFb_results_df.csv", index=False)

# Save eval_df
eval_df.to_csv("SAFb_eval_df.csv", index=False)
